# NB04b Peptide Identification

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/CSBiology/BIO-BTE-06-L-7/gh-pages?filepath=NB04b_Peptide_Identification.ipynb)

[Download Notebook](https://github.com/CSBiology/BIO-BTE-06-L-7/releases/download/NB04a_NB04b/NB04b_Peptide_Identification.ipynb)

1. Understanding peptide identification from MS2 spectra
2. Matching and scoring of Tandem MS peptide identification
    3. Step 1: Data acquisition and preprocessing
    4. Step 2: Preselecting the peptides of interest
    5. Step 3+4: Matching and Scoring

## Understanding peptide identification from MS2 spectra

Under low-energy fragmentation conditions, peptide fragment patterns are reproducible and, in general, predictable, which allows for an 
amino acid sequence identification according to a fragmentation expectation. Algorithms for peptide identification perform in principle 
three basic tasks:

**(i)** a raw data preprocessing step is applied to the MS/MS spectra to obtain clean peak information. The same signal filtering 
and background subtraction methods are used as discussed in the section of low-level processing. Peak detection, however, may be performed 
differently. Preprocessing of MS/MS spectra focuses on the extraction of the precise m/z of the peak rather than the accurate peak areas. 
The conversion of a peak profile into the corresponding m/z and intensity values reduces the complexity, its representation is termed centroiding. 
To extract the masses for identification in a simple and fast way, peak fitting approaches are used. These approaches take either the most intense 
point of the peak profile, fit a Lorentzian distribution to the profile, or use a quadratic fit. 

**(ii)** Spectrum information and possible amino acid sequences assignments are evaluated. 

**(iii)** The quality of the match between spectrum and possible sequences is scored.

Even though the three steps roughly describe the basic principle of algorithms used for peptide sequence identification, most implementations 
show differences in individual steps which can lead to major changes in the outcome. Therefore, it has been proven useful to utilize more than 
one algorithm for a robust and thorough identification. Due to their major difference in identification strategies and prerequisites, 
identification algorithms are normally classified into three categories: (i) *de novo* peptide sequencing, (ii) peptide sequence-tag (PST) 
searching, and (iii) uninterpreted sequence searching. However, in this notebook we focus on the explanation of (iii) uninterpreted sequence 
searching, the most frequently used methods.
## Matching and scoring of Tandem MS peptide identification

![](https://raw.githubusercontent.com/CSBiology/BIO-BTE-06-L-7/main/docs/img/ComputationalProteinIdentification.png)

**Figure 5: Process of computational identification of peptides from their fragment spectra**

Previously we learned, that peptides fragment to create patterns characteristic of a specific amino acid sequence. These patterns are reproducible and, in general, 
predictable taking the applied fragmentation method into account. This can be used for computational identification of peptides from their fragment spectra. 
This process can be subdivided into 5 main steps: spectrum preprocessing, selection of possible sequences, generating theoretical spectra, matching and scoring 
(Figure 5). The first step is a preprocessing of the experimental spectra and is done to reduce noise. Secondly, all possible amino acid 
sequences are selected which match the particular precursor peptide mass. The possible peptides can but do not need to be restricted to a particular organism. 
A theoretical spectrum is predicted for each of these amino acid sequences. Matching and scoring is performed by comparing experimental spectra to their predicted 
corresponding theoretical spectra. The score function measures the closeness of fit between the experimental acquired and theoretical spectrum. There are many 
different score functions, which can be considered as the main reason of different identifications considering different identification algorithm. The most 
natural score function is the cross correlation score (xcorr) used by the commercially available search tool SEQUEST. One of the reasons the xcorr is so 
sensitive is because it involves a correction factor that assesses the background correlation for each acquired spectrum and the theoretically predicted 
spectrum from sequences within a database. To compute this correction factor, a measure of similarity is calculated at different offsets between a preprocessed 
mass spectrum and a theoretical spectrum. The final xcorr is then the correlation at zero offset minus the mean correlation from all the individual offsets.

Thus, the correlation function measures the coherence of two signals by, in effect, translating one signal across the other. This can be mathematically 
achieved using the formula for cross-correlation in the form for discrete input signals:

***Equation 5***

![](https://latex.codecogs.com/png.latex?\large&space;R_{\tau}&space;=&space;\sum_{i=0}^{n-1}x_{i}\cdot%20y_{i&plus;\tau})

where x(i) is a peak of the reconstructed spectrum at position i and y(i) is a peak of the experimental spectrum. The displacement value 𝜏 
is the amount by which the signal is offset during the translation and is varied over a range of values. If two signals are the same, the correlation 
function should have its maxima at 𝜏=0, where there is no offset. The average of the offset-correlation is seen as the average background correlation 
and needs to be subtracted from the correlation at 𝜏=0. It follows: 

***Equation 6***

![](https://latex.codecogs.com/png.latex?xcorr&space;=&space;R_{0}&space;-&space;\frac{(\sum&space;\begin{matrix}&space;\tau=&plus;offeset\\&space;\tau=-offeset\end{matrix}R_{\tau})}{2*offset&plus;1})

In practice many theoretical spectra have to be matched again a single experimental spectrum. Therefore, the calculation can be speed up by reformulating Equation 5 and Equation 6 and introduce a preprocessing step, which is independent of the predicted spectra.

***Equation 7***

![](https://latex.codecogs.com/png.latex?xcorr&space;=&space;x_{0}\cdot&space;y_{0}&space;-&space;\frac{(\sum&space;\begin{matrix}&space;\tau=&plus;offeset\\&space;\tau=-offeset\end{matrix}x_{0}\cdot&space;y_{\tau})}{2*offset&plus;1})

For the preprocessed experimental spectrum y' it follows:

***Equation 8***

![](https://latex.codecogs.com/png.latex?xcorr&space;=&space;x_{0}\cdot&space;y`)

where:

***Equation 9***

![](https://latex.codecogs.com/png.latex?y'&space;=&space;y_{0}&space;-&space;\frac{(\sum&space;\begin{matrix}&space;\tau=&plus;offeset\\&space;\tau=-offeset\end{matrix}x_{0}\cdot&space;y_{\tau})}{2*offset&plus;1})

Matching a measured spectrum against chlamy database


In [1]:
#r "nuget: BioFSharp, 2.0.0-beta4"
#r "nuget: BioFSharp.IO, 2.0.0-beta4"
#r "nuget: Plotly.NET, 4.2.0"
#r "nuget: BioFSharp.Mz, 0.1.5-beta"
#r "nuget: MzIO, 0.1.7"
#r "nuget: MzIO.Processing, 0.1.2"
#r "nuget: MzLite, 1.1.0"
#r "nuget: Plotly.NET.Interactive, 4.2.0"
#r "nuget: MzIO.SQL, 0.1.9"


open Plotly.NET
open BioFSharp
open BioFSharp.Mz
open System.IO
open MzIO
open MzIO.Processing
open MzLite
open MzIO.MzSQL
open System.IO
open System.Data
open BioFSharp.Mz.TheoreticalSpectra
open BioFSharp.Mz.ChargeState



Installed Packages BioFSharp, 2.0.0-beta4 BioFSharp.IO, 2.0.0-beta4 BioFSharp.Mz, 0.1.5-beta MzIO, 0.1.7 MzIO.Processing, 0.1.2 MzIO.SQL, 0.1.9 MzLite, 1.1.0 Plotly.NET, 4.2.0 Plotly.NET.Interactive, 4.2.0

Loading extensions from `/home/paulinehans/.nuget/packages/plotly.net.interactive/4.2.0/lib/netstandard2.1/Plotly.NET.Interactive.dll`

### Step 1: Data acquisition and preprocessing

We load  MS2 spectra that are saved in one mzlite file. The first step is to get the data out of the file


In [2]:
//record type
type ms =  
    {
        mzIntensity: float array * float array
        retentiontime: float
        precursorInfo: list<int*float>
        id: string
    }

In [3]:
// Code-Block 1

//read in mzmlite
let directory = __SOURCE_DIRECTORY__
let path = Path.Combine[|directory;"/home/paulinehans/Downloads/qPgr1_1_5 1.mzlite"|]

//set up connection between file and SQL database
let runID = "sample=0"
let mzReader = new MzIO.MzSQL.MzSQL(path)
let peaksReader = new MzIO.MzSQL.MzSQL(path)
let cn = mzReader.Open()
let cn2 = peaksReader.Open()
let transaction = mzReader.BeginTransaction()



//function that returns all nessecary information we need, in this case m/z-ratio and intensity
let getMs2 = mzReader.ReadMassSpectra runID
let extractData =
    getMs2
    |> Seq.choose (fun x ->
        match MzIO.Processing.MassSpectrum.getMsLevel x with
        | 2 ->
            Some {
                id = x.ID
                precursorInfo = []
                retentiontime = MzIO.Processing.MassSpectrum.getScanTime x
                mzIntensity = 
                    PeakArray.mzIntensityArrayOf (
                        peaksReader.ReadSpectrumPeaks x.ID
                    )
            }
        | _ -> None
    )

    |> Seq.toArray

let allInfo = 
    extractData 
    |> Array.map (fun x -> x.id, x.precursorInfo, x.retentiontime, x.mzIntensity)
allInfo

index value 0 (sample=1 period=1 cycle=2174 experiment=2, [], 19.931816666667, (System.Double[], System.Double[])) Item1 sample=1 period=1 cycle=2174 experiment=2 Item2 [ ] HeadOrDefault <null> TailOrNull <null> Head System.InvalidOperationException: The input list was empty.\n at Microsoft.FSharp.Collections.FSharpList`1.get_Head() in D:\a\_work\1\s\src\FSharp.Core\prim-types.fs:line 4319\n at lambda_method172(Closure, FSharpList`1)\n at Microsoft.DotNet.Interactive.Formatting.MemberAccessor`1.GetValueOrE... TargetSite T get_Head() Name get_Head DeclaringType Microsoft.FSharp.Collections.FSharpList<T> ReflectedType Microsoft.FSharp.Collections.FSharpList<T> MemberType Method MetadataToken 100663785 Module FSharp.Core.dll MDStreamVersion 131072 FullyQualifiedName /home/paulinehans/.nuget/packages/microsoft.dotnet-interactive/1.0.632301/tools/net9.0/any/FSharp.Core.dll ModuleVersionId 1023697b-e00e-d2ad-b568-6f799f8eeeea MetadataToken 1 ScopeName FSharp.Core.dll Name FSharp.Core.dll Assembly FSharp.Core, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a ModuleHandle System.ModuleHandle CustomAttributes [ ] IsSecurityCritical True IsSecuritySafeCritical False IsSecurityTransparent False MethodHandle System.RuntimeMethodHandle Value 133874495734960 Attributes Public, HideBySig, SpecialName CallingConvention Standard, HasThis ReturnType T ReturnTypeCustomAttributes T ParameterType T Name <null> HasDefaultValue False DefaultValue RawDefaultValue MetadataToken 134217728 Attributes None Member T get_Head() Position -1 IsIn False IsLcid False IsOptional False IsOut False IsRetval False CustomAttributes [ ] ReturnParameter T ParameterType T Name <null> HasDefaultValue False DefaultValue RawDefaultValue MetadataToken 134217728 Attributes None Member T get_Head() Position -1 IsIn False IsLcid False IsOptional False IsOut False IsRetval False CustomAttributes [ ] IsCollectible False IsGenericMethod False IsGenericMethodDefinition False ContainsGenericParameters True MethodImplementationFlags IL IsAbstract False IsConstructor False IsFinal False IsHideBySig True IsSpecialName True IsStatic False IsVirtual False IsAssembly False IsFamily False IsFamilyAndAssembly False IsFamilyOrAssembly False IsPrivate False IsPublic True IsConstructedGenericMethod False CustomAttributes (empty) Message The input list was empty. Data (empty) InnerException <null> HelpLink <null> Source FSharp.Core HResult -2146233079 StackTrace at Microsoft.FSharp.Collections.FSharpList`1.get_Head() in D:\a\_work\1\s\src\FSharp.Core\prim-types.fs:line 4319
 at lambda_method172(Closure, FSharpList`1)
 at Microsoft.DotNet.Interactive.Formatting.MemberAccessor`1.GetValueOrException(T instance) in D:\a\_work\1\s\src\Microsoft.DotNet.Interactive.Formatting\MemberAccessor{T}.cs:line 58 Tail System.InvalidOperationException: The input list was empty.\n at Microsoft.FSharp.Collections.FSharpList`1.get_Tail() in D:\a\_work\1\s\src\FSharp.Core\prim-types.fs:line 4324\n at lambda_method173(Closure, FSharpList`1)\n at Microsoft.DotNet.Interactive.Formatting.MemberAccessor`1.GetValueOrE... TargetSite Microsoft.FSharp.Collections.FSharpList`1[T] get_Tail() Name get_Tail DeclaringType Microsoft.FSharp.Collections.FSharpList<T> ReflectedType Microsoft.FSharp.Collections.FSharpList<T> MemberType Method MetadataToken 100663786 Module FSharp.Core.dll MDStreamVersion 131072 FullyQualifiedName /home/paulinehans/.nuget/packages/microsoft.dotnet-interactive/1.0.632301/tools/net9.0/any/FSharp.Core.dll ModuleVersionId 1023697b-e00e-d2ad-b568-6f799f8eeeea MetadataToken 1 ScopeName FSharp.Core.dll Name FSharp.Core.dll Assembly FSharp.Core, Version=9.0.0.0, Culture=neutral, PublicKeyToken=b03f5f7f11d50a3a ModuleHandle System.ModuleHandle CustomAttributes [ ] IsSecurityCritical True IsSecuritySafeCritical False IsSecurityTransparent False MethodHandle System.RuntimeMethodHandle Value 133874495734984 Attributes Public, HideBySig, SpecialName CallingConvention Standard, HasThis ReturnTyp

Here, the spectrum is already centroidized as shown in *NB03c\_Centroidisation.ipynb* using the function 
`msPeakPicking`. So we just visualize mass and intensity:


In [4]:
// Code-Block 2

//first we only spectrum as example
let exampleSpectrum = 
    allInfo 
    |> Array.find (fun (ID, precursorInfo, rt, (mz, intensity)) -> 
        ID.Contains("sample=1 period=1 cycle=1574 experiment=5"))

let exampleData1 = 
    exampleSpectrum 
    |> fun (x,y,z,a) -> fst a 

let exampleData2 = 
    exampleSpectrum 
    |> fun (x,y,z,a) -> snd a 


//creating chart, which visualizes m/z & intensity
let ms2Chart =  
    let chart = 
        Chart.Column(values = exampleData2, Keys = exampleData1,  Width = 5.)
        |> Chart.withYAxisStyle("Intensity")
        |> Chart.withXAxisStyle("m/z")
        |> Chart.withSize(1200, 800)
    chart
ms2Chart


<!-- Plotly chart will be drawn inside this DIV -->

In [5]:
// Code-Block 3

let lowerScanLimit = 150.
let upperScanLimit = 1000.

//all we need
let makingDataPretty = 
    allInfo 
    |> Array.map (fun (ID, _, rt, (mz, intensity)) -> ID,rt,  mz, intensity)

//normalization
let preprocessedIntesities = 
    makingDataPretty 
    |> Array.map (fun (ID,rt, mz,int) -> 
        let normalization =
            Mz.PeakArray.zip mz int
            |> (fun pa -> Mz.PeakArray.peaksToNearestUnitDaltonBinVector pa lowerScanLimit upperScanLimit)
            |> (fun pa -> Mz.SequestLike.windowNormalizeIntensities pa 10)
        ID,rt,normalization
    )
preprocessedIntesities


//example Spectrum Chart
let exampleData = 
    preprocessedIntesities
    |> Array.find (fun (ID, rt, vector) -> 
        ID.Contains("sample=1 period=1 cycle=1574 experiment=5"))
preprocessedIntesities
let (id, rt, vec) = exampleData
let keysValues =
    vec
    |> Seq.mapi (fun i intensity ->
        let mz = lowerScanLimit + float i   
        (mz, intensity)
    )
    |>Seq.toArray
    
let exampleChart = 
        Chart.Column (keysValues = keysValues , Width = 1. )
        |> Chart.withSize(1400,800)
    
exampleChart

<!-- Plotly chart will be drawn inside this DIV -->

Now, we will preprocess the experimental spectrum from our example. This is on the one hand to reduce noise, but also to make 
the measured spectrum more like the once we are able to simulate. 


### Step 2: Preselecting the peptides of interest
From our previos notebook *NB02b\_Digestion\_and\_mass\_calculation.ipynb*, we know how to 
calculate all peptide masses that we can expect to be present in *Chlamydomonas reinhardtii*.




In [6]:
// Code-Block 4
let path2 = Path.Combine[|directory;"downloads/Chlamy_JGI5_5_QProt.fasta"|]

let peptideAndMasses = 
    path2
    |> IO.FastA.fromFile BioArray.ofAminoAcidString
    |> Seq.toArray
    |> Array.mapi (fun i fastAItem ->
        Digestion.BioArray.digest Digestion.Table.Trypsin i fastAItem.Sequence
        |> Digestion.BioArray.concernMissCleavages 0 0
    )
    |> Array.concat
    |> Array.map (fun peptide ->
        // calculate mass for each peptide
        peptide.PepSequence, BioSeq.toMonoisotopicMassWith (BioItem.monoisoMass ModificationInfo.Table.H2O) peptide.PepSequence
    )

peptideAndMasses |> Array.head


([Leu; Ala; Ala; ... ], 1408.6860982223602) Item1 [ Leu, Ala, Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Ala, Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 His 8 His Head Ala Tail [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 His 8 His (values) index value 0 Ala 1 Ala 2 Leu 3 Glu 4 His 5 His 6 His 7 His 8 His 9 His Head Ala Tail [ Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 His 8 His Head Ala Tail [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 His 8 His (values) index value 0 Ala 1 Ala 2 Leu 3 Glu 4 His 5 His 6 His 7 His 8 His 9 His (values) index value 0 Ala 1 Ala 2 Ala 3 Leu 4 Glu 5 His 6 His 7 His 8 His 9 His 10 His Head Leu Tail [ Ala, Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Ala, Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Ala TailOrNull [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His Head Ala Tail [ Leu, Glu, His, His, His, His, His, His ] HeadOrDefault Leu TailOrNull [ Glu, His, His, His, His, His, His ] Head Leu Tail [ Glu, His, His, His, His, His, His ] (values) index value 0 Leu 1 Glu 2 His 3 His 4 His 5 His 6 His 7 His (values) index value 0 Ala 1 Leu 2 Glu 3 His 4 His 5 His 6 His 7 Hi

In the previous notebook *NB04a\_Fragmentation\_for\_peptide\_identification.ipynb*, 
we used functions that generate the theoretical series of b- and y-ions from the given peptide. Combined with the function 
`Mz.SequestLike.predictOf` that generates theoretical spectra that fit the Sequest scoring algorithm.


In [7]:
// Code-Block 6

let predictFromSequence peptide charge =
    [
        peptide
        |> Mz.Fragmentation.Series. yOfBioList BioItem.initMonoisoMassWithMemP
        peptide
        |> Mz.Fragmentation.Series.bOfBioList BioItem.initMonoisoMassWithMemP
    ]
    |> List.concat
    |> Mz.SequestLike.predictOf (lowerScanLimit,upperScanLimit) charge 
 

### Step 3+4: Matching and Scoring

In the matching step, we compare theoretical spectra of peptides within our precursor peptide mass range with our measured spectra. 
We get a score which tells us how well the theoretical spectrum fits the given experimental spectrum.


In [8]:
let numbers : ChargeState.ChargeDetermParams =
    {
        ExpectedMinimalCharge = 2
        ExpectedMaximumCharge = 5
        Width = 1.1
        MinIntensity = 0.15
        DeltaMinIntensity = 0.3
        NrOfRndSpectra = 1000
    }

let rnd = new System.Random()

In [9]:
let getPrecursorCharge rnd =
    /// Returns a Sequence containing all MassSpectra of a single MS-run
    let massSpectra = mzReader.ReadMassSpectra(runID) 
    /// Returns a Array that contains all MS1s sorted by their scanTime
    let ms1SortedByScanTime =
        massSpectra
        |> Seq.filter (fun ms -> MassSpectrum.getMsLevel ms = 1)
        |> Seq.map (fun ms -> MassSpectrum.getScanTime ms, ms)
        |> Seq.sortBy fst
        |> Array.ofSeq
    let ms2SortedByScanTime =
        massSpectra
        |> Seq.filter (fun ms -> MassSpectrum.getMsLevel ms = 2)
        |> Seq.map (fun ms -> MassSpectrum.getScanTime ms, ms)
        |> Seq.sortBy fst
        |> Array.ofSeq
    let ms2AssignedToMS1 =
        ms2SortedByScanTime
        |> Array.choose (fun (ms2ScanTime,ms2) ->
            let ms1' =
                ms1SortedByScanTime
                |> Array.tryFindBack (fun (ms1ScanTime,_) ->
                    ms1ScanTime <= ms2ScanTime)
            match ms1' with 
            | Some (_,ms1) -> 
                Some (ms1, ms2)
            | None -> None 
            )
        |> Array.groupBy fst
        |> Array.map (fun (ms1,ms1And2) -> ms1,ms1And2 |> Array.map snd)
    
    let ms2PossibleChargestates =
        ms2AssignedToMS1
        |> Array.mapi (fun i (ms1,ms2s) ->
            //printfn "%i" i
            let mzdata,intensityData = MzIO.Processing.PeakArray.mzIntensityArrayOf (mzReader.ReadSpectrumPeaks(ms1.ID))
            ms2s
            |> Array.filter (fun ms2 -> mzReader.ReadSpectrumPeaks(ms2.ID).Peaks |> Seq.isEmpty = false)
            |> Array.map (fun ms2 ->
                                let assignedCharges = ChargeState.putativePrecursorChargeStatesBy numbers mzdata intensityData ms1.ID ms2.ID (MassSpectrum.getPrecursorMZ ms2)
                                match assignedCharges with
                                | [] ->
                                    [
                                        for i = 2 to 3 do
                                            let precursorMz = (MassSpectrum.getPrecursorMZ ms2)
                                            let mass = Mass.ofMZ precursorMz (float i)
                                            let score = getScore 10 1 100.
                                            yield createAssignedCharge ms1.ID ms2.ID precursorMz i mass 100. score [0.] 0 0. (Set[]) (Some 1.)
                                    ]
                                    |> Array.ofList
                                | _ -> assignedCharges |> Array.ofList
                            )
            )
        |> Array.filter (fun x ->  Array.isEmpty x |> not)
        |> Array.concat
        |> Array.filter (fun (assignedCharges)  -> assignedCharges.Length = 0 |> not)
    ms2PossibleChargestates


let exe = getPrecursorCharge rnd
exe

let collectChargeMz =
    exe
    |> Array.collect (fun (assignedCharges) ->
                    assignedCharges
                    |> Array.map (fun assCh -> assCh.ProductSpecID, (assCh.PrecCharge, assCh.PutMass))
                )
    |> Array.groupBy (fun (ms2Id,assCH) -> ms2Id)
    |> Array.map (fun (x,y) -> x, y |> Array.map snd)
    |> Map.ofArray
collectChargeMz

key value sample=1 period=1 cycle=1300 experiment=2 index value 0 (2, 832.506703234984) Item1 2 Item2 832.506703234984 1 (4, 1665.013406469968) Item1 4 Item2 1665.013406469968 2 (5, 2081.2667580874604) Item1 5 Item2 2081.2667580874604 sample=1 period=1 cycle=1318 experiment=2 index value 0 (2, 832.510872552214) Item1 2 Item2 832.510872552214 1 (4, 1665.021745104428) Item1 4 Item2 1665.021745104428 2 (5, 2081.2771813805352) Item1 5 Item2 2081.2771813805352 sample=1 period=1 cycle=1322 experiment=2 index value 0 (3, 1593.7217872365147) Item1 3 Item2 1593.7217872365147 1 (2, 1062.48119149101) Item1 2 Item2 1062.48119149101 sample=1 period=1 cycle=1334 experiment=2 index value 0 (2, 832.5127568454479) Item1 2 Item2 832.5127568454479 1 (4, 1665.0255136908959) Item1 4 Item2 1665.0255136908959 2 (5, 2081.28189211362) Item1 5 Item2 2081.28189211362 sample=1 period=1 cycle=1351 experiment=2 index value 0 (2, 832.5105913458219) Item1 2 Item2 832.5105913458219 sample=1 period=1 cycle=1364 experiment=2 index value 0 (3, 1677.782935919445) Item1 3 Item2 1677.782935919445 1 (2, 1118.52195727963) Item1 2 Item2 1118.52195727963 sample=1 period=1 cycle=1365 experiment=2 index value 0 (2, 832.5119342787899) Item1 2 Item2 832.5119342787899 1 (4, 1665.0238685575798) Item1 4 Item2 1665.0238685575798 2 (5, 2081.279835696975) Item1 5 Item2 2081.279835696975 sample=1 period=1 cycle=1378 experiment=2 index value 0 (3, 1677.7746067013159) Item1 3 Item2 1677.7746067013159 1 (2, 1118.516404467544) Item1 2 Item2 1118.516404467544 sample=1 period=1 cycle=1379 experiment=2 index value 0 (2, 832.5102148663159) Item1 2 Item2 832.5102148663159 sample=1 period=1 cycle=1396 experiment=2 index value 0 (2, 832.5102201260919) Item1 2 Item2 832.5102201260919 sample=1 period=1 cycle=1413 experiment=2 index value 0 (2, 832.5086660285319) Item1 2 Item2 832.5086660285319 sample=1 period=1 cycle=1416 experiment=2 index value 0 (3, 1636.719953311878) Item1 3 Item2 1636.719953311878 1 (2, 1091.146635541252) Item1 2 Item2 1091.146635541252 sample=1 period=1 cycle=1427 experiment=2 index value 0 (2, 832.5089412868899) Item1 2 Item2 832.5089412868899 sample=1 period=1 cycle=1428 experiment=2 index value 0 (2, 972.507628186126) Item1 2 Item2 972.507628186126 sample=1 period=1 cycle=1429 experiment=2 index value 0 (2, 986.4674267846159) Item1 2 Item2 986.4674267846159 sample=1 period=1 cycle=1439 experiment=2 index value 0 (2, 832.513376738532) Item1 2 Item2 832.513376738532 sample=1 period=1 cycle=1440 experiment=2 index value 0 (2, 972.5054100018999) Item1 2 Item2 972.5054100018999 sample=1 period=1 cycle=1453 experiment=2 index value 0 (2, 832.5105815426259) Item1 2 Item2 832.5105815426259 sample=1 period=1 cycle=1463 experiment=2 index value 0 (3, 1577.714836023369) Item1 3 Item2 1577.714836023369 1 (2, 1051.8098906822459) Item1 2 Item2 1051.8098906822459 sample=1 period=1 cycle=1466 experiment=2 index value 0 (2, 1122.506985594014) Item1 2 Item2 1122.506985594014 ... (more)

In [10]:
let wholeInfo = 
    preprocessedIntesities
    |> Array.map (fun (id,rt,ms2) -> 
        id,rt, ms2,(collectChargeMz.TryFind(id)).Value
    )
wholeInfo

index value 0 (sample=1 period=1 cycle=2174 experiment=2, 19.931816666667, vector [0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1144303929;0;0.06491106094;0;0;0;0;0;0;0;0;0.08050273973;0;0;0.08084937704;0;0;0;0;0;0;0;0.06669996527;0;0;0;0;0;1;0.2282715197;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.06883138284;0;0;0;0;0... Item1 sample=1 period=1 cycle=2174 experiment=2 Item2 19.931816666667 Item3 [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (more) ] InternalValues [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Values [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] OpsData Some(FSharp.Stats.Instances+FloatNumerics@116) Value FSharp.Stats.Instances+FloatNumerics@116 Length 850 NumRows 850 ElementOps FSharp.Stats.Instances+FloatNumerics@116 DebugDisplay vector [0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1144303929;0;0.06491106094;0;0;0;0;0;0;0;0;0.08050273973;0;0;0.08084937704;0;0;0;0;0;0;0;0.06669996527;0;0;0;0;0;1;0.2282715197;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.06883138284;0;0;0;0;0;0;0;0.6276938903;0.1126905805;0;0;0;0;0.1483438933;0;0;0;0;0... Capacity 361 MaxCapacity 2147483647 Length 361 StructuredDisplayAsArray [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Details [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Transpose [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (more) ] InternalValues [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Values [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] OpsData Some(FSharp.Stats.Instances+FloatNumerics@116) Value FSharp.Stats.Instances+FloatNumerics@116 Length 850 NumCols 850 ElementOps FSharp.Stats.Instances+FloatNumerics@116 DebugDisplay rowvec [0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1144303929;0;0.06491106094;0;0;0;0;0;0;0;0;0.08050273973;0;0;0.08084937704;0;0;0;0;0;0;0;0.06669996527;0;0;0;0;0;1;0.2282715197;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.06883138284;0;0;0;0;0;0;0;0.6276938903;0.1126905805;0;0;0;0;0.1483438933;0;0;0;0;0... Capacity 361 MaxCapacity 2147483647 Length 361 StructuredDisplayAsArray [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Details [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Transpose [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (more) ] InternalValues [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Values [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] OpsData Some(FSharp.Stats.Instances+FloatNumerics@116) Value FSharp.Stats.Instances+FloatNumerics@116 Length 850 NumRows 850 ElementOps FSharp.Stats.Instances+FloatNumerics@116 DebugDisplay vector [0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1144303929;0;0.06491106094;0;0;0;0;0;0;0;0;0.08050273973;0;0;0.08084937704;0;0;0;0;0;0;0;0.06669996527;0;0;0;0;0;1;0.2282715197;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.06883138284;0;0;0;0;0;0;0;0.6276938903;0.1126905805;0;0;0;0;0.1483438933;0;0;0;0;0... Capacity 361 MaxCapacity 2147483647 Length 361 StructuredDisplayAsArray [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Details [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Transpose [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (more) ] InternalValues [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] Values [ 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0 ... (830 more) ] OpsData Some(FSharp.Stats.Instances+FloatNumerics@116) Length 850 NumCols 850 ElementOps FSharp.Stats.Instances+FloatNumerics@116 DebugDisplay rowvec [0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.1144303929;0;0.06491106094;0;0;0;0;0;0;0;0;0.08050273973;0;0;0.08084937704;0;0;0;0;0;0;0;0.06669996527;0;0;0;0;0;1;0.2282715197;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0;0.06883138284;0;0;0

In [17]:
// Code-Block 7

// For each MS2 entry, keep only spectra where at least one peptide candidate can be scored.
let spectras = 
    wholeInfo 
    |> Array.choose (fun (ID,rt, vec,possibleChargesAndMasses) -> 
        // For this spectrum, evaluate each possible precursor charge/mass pair.
        possibleChargesAndMasses
        |> Array.choose (fun (charge,mass) ->
            // Score peptide candidates for one charge state and keep the top-scoring hit.
            let bestScoredPeptideOneCharge = 
                peptideAndMasses
                // Coarse precursor mass filter to reduce the search space.
                |> Array.filter (fun (_,massTheo) -> abs(massTheo-mass) <= 0.5 )
                |> Array.map (fun (sequence,mass) ->
                    // Generate theoretical fragments and compare them to the measured spectrum.
                    let theoSpec = predictFromSequence sequence charge
                    sequence,charge,mass,BioFSharp.Mz.SequestLike.scoreSingle theoSpec vec
                )
                // Rank candidates by score and keep only the best one for this charge.
                |> Array.sortByDescending (fun (_,_,_,score) -> score)
                |> Array.tryHead
            // Convert successful hit to output tuple; skip charge states with no candidates.
            match bestScoredPeptideOneCharge with
            | Some (aaList,charge,mass,score) -> 
                Some (ID, rt,aaList |> BioList.toString,charge,mass,score)
            | None -> None

        )
        // From all tested charge states, keep the single best hit per spectrum.
        |> Array.sortByDescending (fun (_,_,_,_,_,score) -> score)
        |> Array.tryHead
    )
spectras



index value 0 (sample=1 period=1 cycle=2174 experiment=2, 19.931816666667, LLFEALK, 2, 832.5058404091601, 7.463680414533675) Item1 sample=1 period=1 cycle=2174 experiment=2 Item2 19.931816666667 Item3 LLFEALK Item4 2 Item5 832.5058404091601 Item6 7.463680414533675 1 (sample=1 period=1 cycle=2160 experiment=2, 19.84665, LVFPEEVLPR, 2, 1197.6757592657698, 9.395028227159235) Item1 sample=1 period=1 cycle=2160 experiment=2 Item2 19.84665 Item3 LVFPEEVLPR Item4 2 Item5 1197.6757592657698 Item6 9.395028227159235 2 (sample=1 period=1 cycle=2158 experiment=2, 19.82215, LLFEALK, 2, 832.5058404091601, 7.302567421626155) Item1 sample=1 period=1 cycle=2158 experiment=2 Item2 19.82215 Item3 LLFEALK Item4 2 Item5 832.5058404091601 Item6 7.302567421626155 3 (sample=1 period=1 cycle=2144 experiment=2, 19.737, LVFPEEVLPR, 2, 1197.6757592657698, 9.785705961420422) Item1 sample=1 period=1 cycle=2144 experiment=2 Item2 19.737 Item3 LVFPEEVLPR Item4 2 Item5 1197.6757592657698 Item6 9.785705961420422 4 (sample=1 period=1 cycle=2143 experiment=2, 19.7175, LLFEALK, 2, 832.5058404091601, 6.407189581270389) Item1 sample=1 period=1 cycle=2143 experiment=2 Item2 19.7175 Item3 LLFEALK Item4 2 Item5 832.5058404091601 Item6 6.407189581270389 5 (sample=1 period=1 cycle=2129 experiment=2, 19.632166666667, LVFPEEVLPR, 2, 1197.6757592657698, 10.466036162586288) Item1 sample=1 period=1 cycle=2129 experiment=2 Item2 19.632166666667 Item3 LVFPEEVLPR Item4 2 Item5 1197.6757592657698 Item6 10.466036162586288 6 (sample=1 period=1 cycle=2128 experiment=2, 19.612683333333, LLFEALK, 2, 832.5058404091601, 7.050198859953565) Item1 sample=1 period=1 cycle=2128 experiment=2 Item2 19.612683333333 Item3 LLFEALK Item4 2 Item5 832.5058404091601 Item6 7.050198859953565 7 (sample=1 period=1 cycle=2121 experiment=2, 19.562683333333, AVSLVLPSLK, 2, 1025.64848188989, 9.561336996692171) Item1 sample=1 period=1 cycle=2121 experiment=2 Item2 19.562683333333 Item3 AVSLVLPSLK Item4 2 Item5 1025.64848188989 Item6 9.561336996692171 8 (sample=1 period=1 cycle=2117 experiment=3, 19.52785, LVFPEEVLPR, 2, 1197.6757592657698, 9.654704354690317) Item1 sample=1 period=1 cycle=2117 experiment=3 Item2 19.52785 Item3 LVFPEEVLPR Item4 2 Item5 1197.6757592657698 Item6 9.654704354690317 9 (sample=1 period=1 cycle=2116 experiment=3, 19.508183333333, VPLILGIWGGK, 2, 1151.7066654729101, 9.77618365799611) Item1 sample=1 period=1 cycle=2116 experiment=3 Item2 19.508183333333 Item3 VPLILGIWGGK Item4 2 Item5 1151.7066654729101 Item6 9.77618365799611 10 (sample=1 period=1 cycle=2116 experiment=2, 19.501516666667, LLFEALK, 2, 832.5058404091601, 7.1639462178148) Item1 sample=1 period=1 cycle=2116 experiment=2 Item2 19.501516666667 Item3 LLFEALK Item4 2 Item5 832.5058404091601 Item6 7.1639462178148 11 (sample=1 period=1 cycle=2102 experiment=3, 19.422516666667, LVFPEEVLPR, 2, 1197.6757592657698, 9.877081908386863) Item1 sample=1 period=1 cycle=2102 experiment=3 Item2 19.422516666667 Item3 LVFPEEVLPR Item4 2 Item5 1197.6757592657698 Item6 9.877081908386863 12 (sample=1 period=1 cycle=2101 experiment=3, 19.403016666667, VPLILGIWGGK, 2, 1151.7066654729101, 8.861810596587054) Item1 sample=1 period=1 cycle=2101 experiment=3 Item2 19.403016666667 Item3 VPLILGIWGGK Item4 2 Item5 1151.7066654729101 Item6 8.861810596587054 13 (sample=1 period=1 cycle=2101 experiment=2, 19.39635, LLFEALK, 2, 832.5058404091601, 7.196220380333358) Item1 sample=1 period=1 cycle=2101 experiment=2 Item2 19.39635 Item3 LLFEALK Item4 2 Item5 832.5058404091601 Item6 7.196220380333358 14 (sample=1 period=1 cycle=2100 experiment=2, 19.383516666667, AVSLVLPSLK, 2, 1025.64848188989, 9.634386283500437) Item1 sample=1 period=1 cycle=2100 experiment=2 Item2 19.383516666667 Item3 AVSLVLPSLK Item4 2 Item5 1025.64848188989 Item6 9.634386283500437 15 (sample=1 period=1 cycle=2090 experiment=4, 19.318183333333, LVFPEEVLPR, 2, 1197.6757592657698, 10.236493462068053) Item1 sample=1 period=1 cycle=2090 experiment=4 Item2 19.318183333333 Item3 LVFPEEVLPR Item4 2 

So short recap: we compared all experimental spectra in the mzlite file with theoretical spectra from the FASTA file. Now we take the best matching scores. This will tell us, the most probable peptide sequence.

Simply lovely its done but notice however, that in a real world we would need to 
relate our score to the complete data set to get an idea of the overall quality and which numerical value we could trust. 

The last thing we do is to save our data in a .tsv file 

In [13]:

let tableee = 
    bestScore 
    |> Array.map (fun x -> 
        x
    )
tableee

Error: input.fsx (3,5)-(3,14) typecheck error The value or constructor 'bestScore' is not defined. Maybe you want one of the following:
   getScore

## Questions

1. How does the Chart change, when you change the value of 'numberofwindows' from 10 to 20?
  What does this parameter specify? (Code-Block 3)
2. What is the rational behind the normalization of measured spectra?
3. Why are we adding the mass of water to the peptide sequence? (BioItem.monoisoMass ModificationInfo.Table.H2O) (Code-Block 4)
4. In code block 6 you get a raw estimate on how many peptide sequences are candidates for a match, when given a certain mz.
Given that one MS run can consist of up to 120.000 ms2 spectra, how many peptide spectrum matches do you have to perform?
What does that mean for the performance of the application? Which parameters influence the size of the "search space"? (Code-Block 6)
5. What happens when you choose different lower and upper scan limits?
6. Visualize the distribution of scores using a histogram. (Code-Block 7)
